# Lab 3 — European Rain Forecast: Score Your Model Locally

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-model.ipynb)

A forecast submitted to challenge 177 is scored 48 hours later. This
notebook replays the challenge on the past instead: every forecast
issued from 2025-01-01 to 2026-03-01, as the exact request the
challenge sends, scored the way the challenge scores it.

It scores two agents, the city × month climatology of the agent
notebook and a random forest, then leaves the model to you.

---

## 1. Setup and download

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and scikit-learn; the only install
is the ML-Arena client.

In [1]:
!pip install -q mlarena-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 2.1 MB/s eta 0:00:00


In [3]:
import os

MLARENA_API_KEY = "mlk_user_2f53a88429062237_92109197639f19826cb75264f44f77d6"

import mlarena
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor

client = mlarena.connect(api_key=MLARENA_API_KEY)
CHALLENGE_ID = 177

In [4]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

weather_europe_2020_2026.csv.gz 28.2 MB


---

## 2. The data, as the challenge sees it

The challenge sends the cities in its own fixed order, not the
file's alphabetical one. `X` is the whole file as an array (hours,
cities, features) in that order, so a request is a slice of it.

In [5]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

(54096, 45, 8) 2020-01-01 00:00:00+00:00 -> 2026-03-03 23:00:00+00:00


Models learn from **2020–2024**. The replay covers the forecasts
issued from **2025-01-01**, every 6 hours at the UTC hours the
challenge uses, up to the last one whose +48 h is in the file.
`issue_times(start, end)` keeps the issue hours whose +48 h falls
before `end`, so no training target reaches into the replay.

In [6]:
HORIZONS = [6, 48]
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours
SPLIT = hours.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))


def issue_times(start, end):
    """Issue hours in [start, end) with 48 h of history and their +48 h."""
    i = np.arange(max(start, 47), end - max(HORIZONS))
    return i[np.isin(hours.hour[i], ISSUE_HOURS)]


TRAIN = issue_times(0, SPLIT)
TEST = issue_times(SPLIT, len(hours))
print(len(TRAIN), "training issue times |", len(TEST), "replayed,",
      hours[TEST[0]], "->", hours[TEST[-1]])

7293 training issue times | 1700 replayed, 2025-01-01 05:00:00+00:00 -> 2026-03-01 23:00:00+00:00


---

## 3. The request and the local score

`make_request(i)` builds what the challenge sends when hour `i` is
the issue time (the keys are in the agent notebook, section 1): the
48 hours ending at `i`, the 45 cities in panel order, the 8 features.
The feed never reports `visibility`, and the challenge sends it as 0.
`check` validates a response the way the challenge does, and `crps`
is the score of the analysis notebook, section 9.

In [7]:
import json


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def window(i):
    """The history sent at issue hour i: (cities, 48 h, features)."""
    return np.nan_to_num(X[i - 47:i + 1].transpose(1, 0, 2), nan=0.0)


def make_request(i):
    return {
        "issue_time": iso(hours[i]),
        "timestamps": [iso(t) for t in hours[i - 47:i + 1]],
        "observed": [True] * 48,
        "cities": [{"name": n, "country": c,
                    "lat": float(coords.loc[j, "latitude"]),
                    "lon": float(coords.loc[j, "longitude"])}
                   for j, (n, c) in enumerate(PANEL)],
        "feature_names": FEATURES,
        "history": window(i).tolist(),
        "horizons": HORIZONS,
        "max_members": 100,
    }


def check(response, request):
    """The response as a (cities, horizons, M) array, or raise."""
    json.dumps(response, allow_nan=False)      # JSON, no NaN/inf
    ens = np.asarray(response["rain"], dtype=float)
    shape = (len(request["cities"]), len(request["horizons"]))
    assert ens.ndim == 3 and ens.shape[:2] == shape, ens.shape
    assert 1 <= ens.shape[2] <= request["max_members"], ens.shape
    assert np.isfinite(ens).all()
    return ens


def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # clipped at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread

`evaluate(agent, issues)` replays the challenge: for each issue time,
build the request, call `agent.predict`, and score both horizons
against the rain that fell. One row per run; the score is the mean
of the two CRPS, in mm, lower is better.

Consecutive runs share the weather, and the weather moves every
agent's CRPS together. `compare(a, b)` therefore compares two agents
run by run, and takes its standard error from weekly blocks of 28
runs.

In [8]:
def evaluate(agent, issues=TEST):
    """One row per run: CRPS per horizon (mean over cities), score."""
    rows = []
    for i in issues:
        request = make_request(i)
        ens = check(agent.predict(request), request)
        rows.append({f"CRPS {h}h": crps(ens[:, k], rain[i + h]).mean()
                     for k, h in enumerate(HORIZONS)})
    runs = pd.DataFrame(rows, index=hours[issues])
    runs["score"] = runs.mean(axis=1)
    return runs


def compare(a, b, week=28):
    """Mean of a - b over the same runs, and its weekly-block SE."""
    d = (a["score"] - b["score"]).to_numpy()
    blocks = d[:len(d) // week * week].reshape(-1, week).mean(axis=1)
    return d.mean(), blocks.std(ddof=1) / np.sqrt(len(blocks))

---

## 4. The baseline: city × month climatology

The agent notebook's model, refitted on 2020–2024 only: a table
that has seen the replayed months would score itself on its own
data.

In [9]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M


class Climatology:
    """For each city and valid month, M quantiles of the past rain."""

    def __init__(self, end):
        r, m = rain[:end], month[:end]
        self.table = np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                               for k in range(1, 13)])  # (12, cities, M)

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        months = [(issue + pd.Timedelta(hours=h)).month
                  for h in request["horizons"]]
        members = self.table[np.array(months) - 1]    # (2, cities, M)
        return {"rain": members.transpose(1, 0, 2).tolist()}


runs_clim = evaluate(Climatology(SPLIT))
print(runs_clim.mean().round(4).to_string())

CRPS 6h     0.0760
CRPS 48h    0.0758
score       0.0759


---

## 5. A random forest

Five features per city, read from the request: the rain now and
over the last 6 hours, the humidity and the cloud cover now, and the
month. The same `features` function builds the training rows from
`window(i)` and reads the live request, so the model sees at
prediction exactly what it saw in training.

In [10]:
RAIN, HUMIDITY, CLOUDS = (FEATURES.index(f)
                          for f in ("rain", "humidity", "clouds"))


def features(history, issue):
    """(cities, 5) features from a history (cities, 48 h, features)."""
    r = history[:, :, RAIN]
    return np.column_stack([
        r[:, -1],                          # rain now
        r[:, -6:].sum(axis=1),             # rain over the last 6 h
        history[:, -1, HUMIDITY],
        history[:, -1, CLOUDS],
        np.full(len(history), issue.month),
    ])


F = np.concatenate([features(window(i), hours[i]) for i in TRAIN])
print(F.shape, "training rows: issue times x cities")

(328185, 5) training rows: issue times x cities


One forest per horizon. A forest is an ensemble already: each of its
30 trees gives one member.

In [11]:
forests = {
    h: RandomForestRegressor(n_estimators=30, min_samples_leaf=100,
                             n_jobs=-1, random_state=0)
    .fit(F, np.concatenate([rain[i + h] for i in TRAIN]))
    for h in HORIZONS
}


class Forest:
    def __init__(self, forests):
        self.forests = forests

    def predict(self, request):
        F = features(np.asarray(request["history"]),
                     pd.Timestamp(request["issue_time"]))
        members = [np.stack([tree.predict(F)
                             for tree in self.forests[h].estimators_],
                            axis=-1)
                   for h in request["horizons"]]    # 2 x (cities, 30)
        return {"rain": np.stack(members, axis=1).tolist()}


runs_rf = evaluate(Forest(forests))
print(runs_rf.mean().round(4).to_string())
print("forest - climatology: %+.4f ± %.4f" % compare(runs_rf, runs_clim))

CRPS 6h     0.1020
CRPS 48h    0.1132
score       0.1076
forest - climatology: +0.0317 ± 0.0006


**Finding.** The forest scores 0.1076 (0.1020 at +6 h, 0.1132 at
+48 h): 0.0317 ± 0.0006 worse than the climatology, and worse than
forecasting 0 mm everywhere (0.0804, the analysis notebook). It reads
the weather; the climatology reads nothing.

**Question.** Why? Each tree predicts the mean of its leaf. Compare
the forest's members with the climatology's for one city and one run
— how many are 0, how far apart they are — and say which term of the
CRPS the forest loses on.

*Your answer:*

---

## 6. Your turn

Build features, fit a model, and score it here before you submit it.

- **Features.** Anything computed from the request. The analysis
  notebook lists the candidates: rain in the city upwind (section 7),
  the solar hour in summer (section 5), a 6-hour change with the time
  of day beside it (section 8). Write them in `features`: `window(i)`
  gives it in training exactly what the request gives it live.
- **Model.** Anything whose `predict(request)` returns (cities, 2, M)
  members, 1 ≤ M ≤ 100. The members are a distribution: a spike at
  0 and a long tail. A hurdle model (a classifier for P(wet), a
  distribution of amounts given wet, its quantiles as members) or one
  quantile regressor per level both produce one.
- **Evaluate.** `evaluate(agent, issues)`, then `compare` with the
  climatology on the same runs. A gap is real when it is several
  times its ±.

The bars, over the same replay: the climatology, 0.0759; the
challenge's starter agent (the template on the challenge page),
0.0746; a +6 h climatology split by rain in the last 3 hours, with
this climatology at +48 h, 0.0742.

Every look at the replay is a look at the test period. Tune on
2024, fitted on 2020–2023 (the cell below), and replay 2025–2026 once
per model you would submit, refitted on 2020–2024. To submit, write
`agent.py` as in the agent notebook, save the models next to it
(`joblib.dump`), and submit with `runtime_id=182`, the runtime with
scikit-learn, LightGBM and XGBoost.

In [12]:
VALID = hours.searchsorted(pd.Timestamp("2024-01-01", tz="UTC"))
FIT = issue_times(0, VALID)       # 2020—2023 的发布时刻
VAL = issue_times(VALID, SPLIT)   # 2024 的发布时刻

clim_val = evaluate(Climatology(VALID), VAL)
print("2024 气候基线：")
print(clim_val.mean().round(4).to_string())


class RecentRainClimatology:
    """+6 h 按最近三小时是否下雨分组；+48 h 使用城市×月份分位数。"""

    def __init__(self, end, fit_issues, members=100, min_count=30):
        self.levels = (np.arange(members) + 0.5) / members
        self.city_index = {
            f"{name}|{country}": j
            for j, (name, country) in enumerate(PANEL)
        }

        # 先从训练期的所有小时计算城市×月份的基础分布。
        past_rain = rain[:end]
        past_month = month[:end]
        self.base = np.stack([
            np.quantile(
                past_rain[past_month == k],
                self.levels,
                axis=0,
            ).T
            for k in range(1, 13)
        ])  # (12, 45, members)

        # 每个训练发布时间、每座城市最近三小时是否曾下雨。
        recently_wet = np.stack([
            rain[fit_issues - lag] >= 0.1
            for lag in (0, 1, 2)
        ]).any(axis=0)  # (训练发布时间, 45)

        target_rain = rain[fit_issues + 6]
        target_month = month[fit_issues + 6]

        # 为 +6 h 分别建立「此前干」与「此前湿」的经验分布。
        self.short = np.empty((12, len(PANEL), 2, members))
        self.counts = np.zeros((12, len(PANEL), 2), dtype=int)

        for k in range(1, 13):
            in_month = target_month == k

            for j in range(len(PANEL)):
                for state in (0, 1):
                    selected = in_month & (recently_wet[:, j] == bool(state))
                    n = selected.sum()
                    self.counts[k - 1, j, state] = n

                    if n >= min_count:
                        self.short[k - 1, j, state] = np.quantile(
                            target_rain[selected, j],
                            self.levels,
                        )
                    else:
                        self.short[k - 1, j, state] = self.base[k - 1, j]

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        history = np.asarray(request["history"], dtype=float)

        # history 的形状是 (城市, 48 小时, 8 个变量)。
        recently_wet = (
            history[:, -3:, RAIN] >= 0.1
        ).any(axis=1)

        forecasts = []

        for city_pos, city in enumerate(request["cities"]):
            key = f"{city['name']}|{city['country']}"
            j = self.city_index[key]
            by_horizon = []

            for h in request["horizons"]:
                target_month = (issue + pd.Timedelta(hours=h)).month - 1

                if h == 6:
                    state = int(recently_wet[city_pos])
                    values = self.short[target_month, j, state]
                elif h == 48:
                    values = self.base[target_month, j]
                else:
                    raise ValueError(f"尚未定义 {h} 小时的预测")

                by_horizon.append(values.tolist())

            forecasts.append(by_horizon)

        return {"rain": forecasts}


mine = RecentRainClimatology(VALID, FIT)

print("训练发布时间：", len(FIT), "；2024 验证发布时间：", len(VAL))
print("分组样本数范围：", mine.counts.min(), "至", mine.counts.max())

runs_mine = evaluate(mine, VAL)

print("2024 新模型：")
print(runs_mine.mean().round(4).to_string())

difference, se = compare(runs_mine, clim_val)
print(f"新模型－气候基线：{difference:+.4f} ± {se:.4f}")
# Your model: fit on FIT only, then
# mine = ...                          # anything with .predict(request)
# runs_mine = evaluate(mine, VAL)
# print("mine - climatology: %+.4f ± %.4f" % compare(runs_mine, clim_val))

2024 气候基线：
CRPS 6h     0.0811
CRPS 48h    0.0808
score       0.0809
训练发布时间： 5829 ；2024 验证发布时间： 1456
分组样本数范围： 0 至 496
2024 新模型：
CRPS 6h     0.0782
CRPS 48h    0.0808
score       0.0795
新模型－气候基线：-0.0015 ± 0.0002


In [13]:
diagnostic = pd.DataFrame({
    "month": np.asarray(hours[VAL + 6].month),
    "baseline_6h": clim_val["CRPS 6h"].to_numpy(),
    "model_6h": runs_mine["CRPS 6h"].to_numpy(),
})
diagnostic["difference_6h"] = (
    diagnostic["model_6h"] - diagnostic["baseline_6h"]
)

by_month = diagnostic.groupby("month").agg(
    runs=("difference_6h", "size"),
    baseline_6h=("baseline_6h", "mean"),
    model_6h=("model_6h", "mean"),
    difference_6h=("difference_6h", "mean"),
)

print("按目标月份比较 +6 小时 CRPS：")
print(by_month.round(4).to_string())

sparse = mine.counts < 30
print("\n分组总数：", mine.counts.size)
print("触发基础分布回退的组数：", sparse.sum())
print("其中最近三小时干的组数：", sparse[:, :, 0].sum())
print("其中最近三小时湿的组数：", sparse[:, :, 1].sum())
print("各状态样本数的中位数：",
      np.median(mine.counts[:, :, 0]),
      np.median(mine.counts[:, :, 1]))

按目标月份比较 +6 小时 CRPS：
       runs  baseline_6h  model_6h  difference_6h
month                                            
1       123       0.0559    0.0531        -0.0028
2       116       0.0789    0.0746        -0.0043
3       124       0.0842    0.0803        -0.0039
4       120       0.0769    0.0748        -0.0022
5       124       0.1016    0.0988        -0.0028
6       120       0.0944    0.0928        -0.0015
7       124       0.0746    0.0736        -0.0010
8       124       0.0670    0.0661        -0.0009
9       120       0.1074    0.1029        -0.0044
10      124       0.0992    0.0935        -0.0057
11      120       0.0636    0.0601        -0.0035
12      117       0.0696    0.0668        -0.0027

分组总数： 1080
触发基础分布回退的组数： 13
其中最近三小时干的组数： 0
其中最近三小时湿的组数： 13
各状态样本数的中位数： 383.0 100.5


In [14]:
class RainAndCloudClimatology(RecentRainClimatology):
    """在最近三小时雨况之外，检验当前云量对 +6 h 的增量价值。"""

    def __init__(self, end, fit_issues, cloud_threshold=60, min_cloud_count=50):
        super().__init__(end, fit_issues)

        self.cloud_threshold = cloud_threshold
        self.short_cloud = np.empty(
            (12, len(PANEL), 2, 2, len(self.levels))
        )
        self.cloud_counts = np.zeros(
            (12, len(PANEL), 2, 2), dtype=int
        )

        recently_wet = np.stack([
            rain[fit_issues - lag] >= 0.1
            for lag in (0, 1, 2)
        ]).any(axis=0)

        cloud_index = FEATURES.index("clouds")
        cloud_now = np.nan_to_num(
            X[fit_issues, :, cloud_index], nan=0.0
        )
        cloudy = cloud_now >= cloud_threshold

        target_rain = rain[fit_issues + 6]
        target_month = month[fit_issues + 6]

        for k in range(1, 13):
            in_month = target_month == k

            for j in range(len(PANEL)):
                for rain_state in (0, 1):
                    for cloud_state in (0, 1):
                        selected = (
                            in_month
                            & (recently_wet[:, j] == bool(rain_state))
                            & (cloudy[:, j] == bool(cloud_state))
                        )
                        n = selected.sum()
                        self.cloud_counts[
                            k - 1, j, rain_state, cloud_state
                        ] = n

                        if n >= min_cloud_count:
                            values = np.quantile(
                                target_rain[selected, j],
                                self.levels,
                            )
                        else:
                            # 样本太少：使用第一步已经训练好的分布。
                            values = self.short[k - 1, j, rain_state]

                        self.short_cloud[
                            k - 1, j, rain_state, cloud_state
                        ] = values

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        history = np.asarray(request["history"], dtype=float)

        rain_index = request["feature_names"].index("rain")
        cloud_index = request["feature_names"].index("clouds")

        recently_wet = (
            history[:, -3:, rain_index] >= 0.1
        ).any(axis=1)
        cloudy = (
            history[:, -1, cloud_index] >= self.cloud_threshold
        )

        forecasts = []

        for city_pos, city in enumerate(request["cities"]):
            key = f"{city['name']}|{city['country']}"
            j = self.city_index[key]
            by_horizon = []

            for h in request["horizons"]:
                target_month = (
                    issue + pd.Timedelta(hours=h)
                ).month - 1

                if h == 6:
                    values = self.short_cloud[
                        target_month,
                        j,
                        int(recently_wet[city_pos]),
                        int(cloudy[city_pos]),
                    ]
                elif h == 48:
                    values = self.base[target_month, j]
                else:
                    raise ValueError(f"尚未定义 {h} 小时的预测")

                by_horizon.append(values.tolist())

            forecasts.append(by_horizon)

        return {"rain": forecasts}


mine_cloud = RainAndCloudClimatology(VALID, FIT)

print(
    "云量细分后少于 50 个样本的组数：",
    (mine_cloud.cloud_counts < 50).sum(),
    "/",
    mine_cloud.cloud_counts.size,
)

runs_cloud = evaluate(mine_cloud, VAL)

print("2024 云量模型：")
print(runs_cloud.mean().round(4).to_string())

difference, se = compare(runs_cloud, runs_mine)
print(f"云量模型－第一步模型：{difference:+.4f} ± {se:.4f}")

云量细分后少于 50 个样本的组数： 631 / 2160
2024 云量模型：
CRPS 6h     0.0779
CRPS 48h    0.0808
score       0.0793
云量模型－第一步模型：-0.0001 ± 0.0000


In [15]:
difference, se = compare(runs_cloud, runs_mine)
print(f"综合配对差值：{difference:+.7f} ± {se:.7f}")
print(
    "仅 +6 h 的平均差值：",
    f"{(runs_cloud['CRPS 6h'] - runs_mine['CRPS 6h']).mean():+.7f}",
)

# 复现验证请求的「最近三小时湿」与「当前云量高」状态。
val_recently_wet = np.stack([
    rain[VAL - lag] >= 0.1
    for lag in (0, 1, 2)
]).any(axis=0)

cloud_index = FEATURES.index("clouds")
val_cloudy = (
    np.nan_to_num(X[VAL, :, cloud_index], nan=0.0)
    >= mine_cloud.cloud_threshold
)

# 每个 2024 发布时间、每座城市实际落入的训练分组有多少样本。
group_sizes = mine_cloud.cloud_counts[
    month[VAL + 6, None] - 1,
    np.arange(len(PANEL))[None, :],
    val_recently_wet.astype(int),
    val_cloudy.astype(int),
]

print(
    "2024 城市×发布时间中实际触发回退的比例：",
    f"{(group_sizes < 50).mean():.2%}",
)
print(
    "低于 50 个样本的训练组，按[最近干/湿, 云量低/高]排列：\n",
    (mine_cloud.cloud_counts < 50).sum(axis=(0, 1)),
)

综合配对差值：-0.0001428 ± 0.0000434
仅 +6 h 的平均差值： -0.0002856
2024 城市×发布时间中实际触发回退的比例： 5.34%
低于 50 个样本的训练组，按[最近干/湿, 云量低/高]排列：
 [[  5  13]
 [530  83]]


In [16]:
class WithLongMemory(RainAndCloudClimatology):
    """保留 +6 h 云量模型，只测试 +48 h 的最近雨况分组。"""

    def __init__(self, end, fit_issues, min_long_count=30):
        super().__init__(end, fit_issues)

        recently_wet = np.stack([
            rain[fit_issues - lag] >= 0.1
            for lag in (0, 1, 2)
        ]).any(axis=0)

        target_rain = rain[fit_issues + 48]
        target_month = month[fit_issues + 48]

        self.long = np.empty(
            (12, len(PANEL), 2, len(self.levels))
        )
        self.long_counts = np.zeros(
            (12, len(PANEL), 2), dtype=int
        )

        for k in range(1, 13):
            in_month = target_month == k

            for j in range(len(PANEL)):
                for state in (0, 1):
                    selected = (
                        in_month
                        & (recently_wet[:, j] == bool(state))
                    )
                    n = selected.sum()
                    self.long_counts[k - 1, j, state] = n

                    if n >= min_long_count:
                        values = np.quantile(
                            target_rain[selected, j],
                            self.levels,
                        )
                    else:
                        values = self.base[k - 1, j]

                    self.long[k - 1, j, state] = values

    def predict(self, request):
        # 先取得第二步模型的完整预测，保持 +6 h 不变。
        result = super().predict(request)

        history = np.asarray(request["history"], dtype=float)
        rain_index = request["feature_names"].index("rain")
        recently_wet = (
            history[:, -3:, rain_index] >= 0.1
        ).any(axis=1)

        issue = pd.Timestamp(request["issue_time"])

        for city_pos, city in enumerate(request["cities"]):
            key = f"{city['name']}|{city['country']}"
            j = self.city_index[key]

            for horizon_pos, h in enumerate(request["horizons"]):
                if h == 48:
                    target_month = (
                        issue + pd.Timedelta(hours=h)
                    ).month - 1

                    result["rain"][city_pos][horizon_pos] = (
                        self.long[
                            target_month,
                            j,
                            int(recently_wet[city_pos]),
                        ].tolist()
                    )

        return result


mine_long = WithLongMemory(VALID, FIT)
runs_long = evaluate(mine_long, VAL)

print("2024 加入 +48 h 雨况记忆：")
print(runs_long.mean().round(6).to_string())

difference, se = compare(runs_long, runs_cloud)
print(f"相对云量模型：{difference:+.7f} ± {se:.7f}")

print(
    "+6 h 的最大逐次分数差：",
    np.abs(
        runs_long["CRPS 6h"].to_numpy()
        - runs_cloud["CRPS 6h"].to_numpy()
    ).max(),
)

2024 加入 +48 h 雨况记忆：
CRPS 6h     0.077880
CRPS 48h    0.080772
score       0.079326
相对云量模型：+0.0000064 ± 0.0000303
+6 h 的最大逐次分数差： 0.0


In [17]:
class HourlyLongClimatology:
    def __init__(self, base_agent, fit_issues, min_count=80):
        self.base_agent = base_agent
        self.min_count = min_count
        self.tables = {}
        self.counts = {}

        # +48 h 的真实降雨量，以及对应目标时刻的月份、UTC 小时
        targets = fit_issues + 48
        target_rain = rain[targets]
        target_month = month[targets]
        target_hour = hours.hour[targets]

        for m in range(1, 13):
            for h in ISSUE_HOURS:
                selected = (target_month == m) & (target_hour == h)
                n = int(selected.sum())
                self.counts[(m, h)] = n

                if n >= min_count:
                    # 每个城市各取 100 个分位数：(cities, M)
                    self.tables[(m, h)] = np.quantile(
                        target_rain[selected],
                        LEVELS,
                        axis=0
                    ).T

    def predict(self, request):
        # 先取得已保留的云量模型预测，+6 h 原样沿用
        members = np.asarray(
            self.base_agent.predict(request)["rain"],
            dtype=float
        )

        issue = pd.Timestamp(request["issue_time"])
        target = issue + pd.Timedelta(hours=48)
        key = (target.month, target.hour)

        # 样本足够时，仅替换 +48 h 对应的分布
        if key in self.tables:
            members[:, 1, :] = self.tables[key]

        return {"rain": members.tolist()}


mine_hour = HourlyLongClimatology(mine_cloud, FIT)
runs_hour = evaluate(mine_hour, VAL)

print("2024：+48 h 按目标 UTC 小时分组")
print(runs_hour.mean().round(6).to_string())

difference, se = compare(runs_hour, runs_cloud)
print(f"相对云量模型：{difference:+.7f} ± {se:.7f}")

short_gap = np.abs(
    runs_hour["CRPS 6h"].to_numpy()
    - runs_cloud["CRPS 6h"].to_numpy()
).max()
print("＋6 h 最大逐次分数差：", short_gap)

print("各月份、目标时刻的训练样本数：")
print(pd.DataFrame(
    {
        f"{h:02d}:00": [mine_hour.counts[(m, h)] for m in range(1, 13)]
        for h in ISSUE_HOURS
    },
    index=pd.Index(range(1, 13), name="目标月份")
).to_string())

2024：+48 h 按目标 UTC 小时分组
CRPS 6h     0.077880
CRPS 48h    0.080875
score       0.079378
相对云量模型：+0.0000582 ± 0.0000298
＋6 h 最大逐次分数差： 0.0
各月份、目标时刻的训练样本数：
      05:00  11:00  17:00  23:00
目标月份                            
1       120    120    120    121
2       113    113    113    113
3       124    124    124    124
4       120    120    120    120
5       124    124    124    124
6       120    120    120    120
7       124    124    124    124
8       124    124    124    124
9       120    120    120    120
10      124    124    124    124
11      120    120    120    120
12      124    124    124    124


In [18]:
class RainPhaseClimatology:
    def __init__(self, base_agent, fit_issues, min_count=30):
        self.base_agent = base_agent
        self.tables = {}
        self.counts = {}
        self.overrides = 0

        target_month = month[fit_issues + 6]
        target_rain = rain[fit_issues + 6]

        # 与预测时可见的最近三小时一致
        recent_wet = (
            (rain[fit_issues] >= 0.1)
            | (rain[fit_issues - 1] >= 0.1)
            | (rain[fit_issues - 2] >= 0.1)
        )
        raining_now = rain[fit_issues] >= 0.1
        high_cloud = (
            np.nan_to_num(
                X[fit_issues, :, FEATURES.index("clouds")],
                nan=0.0
            ) >= 60
        )

        for m in range(1, 13):
            month_mask = target_month == m
            for city in range(rain.shape[1]):
                # phase=0：最近下过雨，但当前已停
                # phase=1：当前仍在下雨
                for phase in (0, 1):
                    for cloudy in (0, 1):
                        selected = (
                            month_mask
                            & recent_wet[:, city]
                            & (raining_now[:, city] == bool(phase))
                            & (high_cloud[:, city] == bool(cloudy))
                        )
                        key = (m, city, phase, cloudy)
                        n = int(selected.sum())
                        self.counts[key] = n

                        if n >= min_count:
                            self.tables[key] = np.quantile(
                                target_rain[selected, city],
                                LEVELS
                            )

    def predict(self, request):
        members = np.asarray(
            self.base_agent.predict(request)["rain"],
            dtype=float
        )
        history = np.asarray(request["history"], dtype=float)
        rain_col = request["feature_names"].index("rain")
        cloud_col = request["feature_names"].index("clouds")

        recent_wet = (history[:, -3:, rain_col] >= 0.1).any(axis=1)
        raining_now = history[:, -1, rain_col] >= 0.1
        high_cloud = history[:, -1, cloud_col] >= 60

        issue = pd.Timestamp(request["issue_time"])
        target_month = (issue + pd.Timedelta(hours=6)).month

        for city in range(history.shape[0]):
            if not recent_wet[city]:
                continue

            key = (
                target_month,
                city,
                int(raining_now[city]),
                int(high_cloud[city])
            )
            if key in self.tables:
                members[city, 0, :] = self.tables[key]
                self.overrides += 1

        return {"rain": members.tolist()}


mine_phase = RainPhaseClimatology(mine_cloud, FIT)
runs_phase = evaluate(mine_phase, VAL)

print("2024：区分雨仍在下／雨已停")
print(runs_phase.mean().round(6).to_string())

difference, se = compare(runs_phase, runs_cloud)
print(f"相对云量模型：{difference:+.7f} ± {se:.7f}")

long_gap = np.abs(
    runs_phase["CRPS 48h"].to_numpy()
    - runs_cloud["CRPS 48h"].to_numpy()
).max()
print("＋48 h 最大逐次分数差：", long_gap)

print(
    "验证期城市预测中，使用新分组的比例：",
    f"{mine_phase.overrides / (len(VAL) * len(PANEL)):.2%}"
)
print(
    "达到 30 个训练样本的分组：",
    f"{len(mine_phase.tables)} / {len(mine_phase.counts)}"
)

2024：区分雨仍在下／雨已停
CRPS 6h     0.077857
CRPS 48h    0.080759
score       0.079308
相对云量模型：-0.0000114 ± 0.0000274
＋48 h 最大逐次分数差： 0.0
验证期城市预测中，使用新分组的比例： 13.49%
达到 30 个训练样本的分组： 602 / 2160


In [19]:
FEATURE_COLUMNS = [
    "city",
    "season_sin", "season_cos",
    "hour_sin", "hour_cos",
    "rain_now", "rain_1h_ago", "rain_2h_ago",
    "rain_sum_3h", "rain_sum_6h",
    "rain_sum_24h", "rain_sum_48h",
    "rain_change_6h",
    "cloud_now", "cloud_change_6h",
    "humidity_now", "temperature_now", "wind_speed_now",
]


def features_from_history(history, issue_time):
    """从一次请求允许看到的48小时历史，生成每座城市的一行特征。"""
    history = np.nan_to_num(
        np.asarray(history, dtype=float),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    n_cities = history.shape[0]

    rain_history = history[:, :, FEATURES.index("rain")]
    cloud_history = history[:, :, FEATURES.index("clouds")]

    # 时间特征描述的是 +6 h 的预测目标时刻
    target = pd.Timestamp(issue_time) + pd.Timedelta(hours=6)
    season_angle = 2 * np.pi * (target.dayofyear - 1) / 365.25
    hour_angle = 2 * np.pi * target.hour / 24

    result = np.column_stack([
        np.arange(n_cities),
        np.full(n_cities, np.sin(season_angle)),
        np.full(n_cities, np.cos(season_angle)),
        np.full(n_cities, np.sin(hour_angle)),
        np.full(n_cities, np.cos(hour_angle)),
        rain_history[:, -1],
        rain_history[:, -2],
        rain_history[:, -3],
        rain_history[:, -3:].sum(axis=1),
        rain_history[:, -6:].sum(axis=1),
        rain_history[:, -24:].sum(axis=1),
        rain_history.sum(axis=1),
        rain_history[:, -1] - rain_history[:, -7],
        cloud_history[:, -1],
        cloud_history[:, -1] - cloud_history[:, -7],
        history[:, -1, FEATURES.index("humidity")],
        history[:, -1, FEATURES.index("temperature")],
        history[:, -1, FEATURES.index("wind_speed")],
    ])

    return result.astype(np.float32)


# 每个发布时刻产生45行：一座城市一行
feat_fit = np.concatenate(
    [features_from_history(window(i), hours[i]) for i in FIT],
    axis=0
)
feat_val = np.concatenate(
    [features_from_history(window(i), hours[i]) for i in VAL],
    axis=0
)

# 标签是发布6小时后的实际雨量；不作为特征输入
y_fit = rain[FIT + 6].reshape(-1).astype(np.float32)
y_val = rain[VAL + 6].reshape(-1).astype(np.float32)

# 核对训练特征与真实请求中的特征完全一致
sample_issue = FIT[0]
sample_request = make_request(sample_issue)
np.testing.assert_allclose(
    features_from_history(window(sample_issue), hours[sample_issue]),
    features_from_history(
        sample_request["history"],
        sample_request["issue_time"]
    )
)

assert feat_fit.shape == (len(FIT) * len(PANEL), len(FEATURE_COLUMNS))
assert feat_val.shape == (len(VAL) * len(PANEL), len(FEATURE_COLUMNS))
assert np.isfinite(feat_fit).all() and np.isfinite(feat_val).all()

print("训练特征：", feat_fit.shape, "训练标签：", y_fit.shape)
print("验证特征：", feat_val.shape, "验证标签：", y_val.shape)
print("训练期达到0.1 mm的比例：", f"{(y_fit >= 0.1).mean():.2%}")
print("验证期达到0.1 mm的比例：", f"{(y_val >= 0.1).mean():.2%}")
print("训练特征与预测请求一致：通过")

训练特征： (262305, 18) 训练标签： (262305,)
验证特征： (65520, 18) 验证标签： (65520,)
训练期达到0.1 mm的比例： 14.39%
验证期达到0.1 mm的比例： 14.23%
训练特征与预测请求一致：通过


In [20]:
from lightgbm import LGBMClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score

wet_fit = (y_fit >= 0.1).astype(np.int8)
wet_val = (y_val >= 0.1).astype(np.int8)

wet_model = LGBMClassifier(
    n_estimators=180,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=200,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

wet_model.fit(
    feat_fit,
    wet_fit,
    categorical_feature=[0]  # FEATURE_COLUMNS[0] 是 city
)

p_val = wet_model.predict_proba(feat_val)[:, 1]
constant_p = np.full_like(p_val, wet_fit.mean())

print("训练期达到阈值的比例：", f"{wet_fit.mean():.2%}")
print("验证期达到阈值的比例：", f"{wet_val.mean():.2%}")
print("恒定概率的 Brier 分数：",
      f"{brier_score_loss(wet_val, constant_p):.6f}")
print("模型概率的 Brier 分数：",
      f"{brier_score_loss(wet_val, p_val):.6f}")
print("模型 ROC AUC：", f"{roc_auc_score(wet_val, p_val):.4f}")

# 按预测概率从低到高分成十组，检查预测概率与实际频率
calibration = pd.DataFrame({
    "predicted": p_val,
    "observed": wet_val
})
calibration["group"] = (
    pd.qcut(
        calibration["predicted"].rank(method="first"),
        10,
        labels=False
    ) + 1
)
calibration = calibration.groupby("group").agg(
    samples=("observed", "size"),
    mean_probability=("predicted", "mean"),
    actual_frequency=("observed", "mean")
)

print("\n概率从低到高的十组：")
print(calibration.round(4).to_string())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


训练期达到阈值的比例： 14.39%
验证期达到阈值的比例： 14.23%
恒定概率的 Brier 分数： 0.122081
模型概率的 Brier 分数： 0.097484
模型 ROC AUC： 0.8289

概率从低到高的十组：
       samples  mean_probability  actual_frequency
group                                             
1         6552            0.0049            0.0032
2         6552            0.0130            0.0122
3         6552            0.0271            0.0255
4         6552            0.0476            0.0403
5         6552            0.0742            0.0702
6         6552            0.1088            0.0955
7         6552            0.1567            0.1436
8         6552            0.2267            0.2169
9         6552            0.3309            0.3024
10        6552            0.5217            0.5136


In [22]:
class ProbabilityRainAgent:
    def __init__(self, base_agent, wet_model, train_end, min_wet=30):
        self.base_agent = base_agent
        self.wet_model = wet_model
        self.wet_quantiles = {}
        self.fallback_count = 0

        past_rain = rain[:train_end]
        past_month = month[:train_end]

        # 样本不足时，退回该城市所有月份的湿日雨量分布
        city_quantiles = {}
        for city in range(past_rain.shape[1]):
            values = past_rain[:, city]
            values = values[values >= 0.1]
            city_quantiles[city] = np.quantile(values, LEVELS)

        for m in range(1, 13):
            for city in range(past_rain.shape[1]):
                values = past_rain[past_month == m, city]
                values = values[values >= 0.1]

                if len(values) >= min_wet:
                    self.wet_quantiles[(m, city)] = np.quantile(
                        values, LEVELS
                    )
                else:
                    self.wet_quantiles[(m, city)] = city_quantiles[city]
                    self.fallback_count += 1

    def predict(self, request):
        # 保留现有模型的 +48 h 预测
        members = np.asarray(
            self.base_agent.predict(request)["rain"],
            dtype=float
        )

        features = features_from_history(
            request["history"],
            request["issue_time"]
        )

        named_features = pd.DataFrame(
          features,
          columns=self.wet_model.feature_name_
        )
        p_wet = self.wet_model.predict_proba(named_features)[:, 1]

        issue = pd.Timestamp(request["issue_time"])
        target_month = (issue + pd.Timedelta(hours=6)).month

        # LEVELS 是 100 个成员对应的分位点
        for city, probability in enumerate(p_wet):
            probability = float(np.clip(probability, 0.0, 1.0))
            values = np.zeros(len(LEVELS), dtype=float)

            if probability > 0:
                # 混合分布：先有 1-p 的零雨量，再有 p 的湿日雨量
                wet_members = LEVELS > (1.0 - probability)
                conditional_levels = (
                    LEVELS[wet_members] - (1.0 - probability)
                ) / probability

                values[wet_members] = np.interp(
                    conditional_levels,
                    LEVELS,
                    self.wet_quantiles[(target_month, city)]
                )

            members[city, 0, :] = values

        return {"rain": members.tolist()}


mine_probability = ProbabilityRainAgent(
    base_agent=mine_cloud,
    wet_model=wet_model,
    train_end=VALID
)

runs_probability = evaluate(mine_probability, VAL)

print("2024：概率模型 + 历史湿日雨量分布")
print(runs_probability.mean().round(6).to_string())

difference, se = compare(runs_probability, runs_cloud)
print(f"相对云量模型：{difference:+.7f} ± {se:.7f}")

long_gap = np.abs(
    runs_probability["CRPS 48h"].to_numpy()
    - runs_cloud["CRPS 48h"].to_numpy()
).max()
print("＋48 h 最大逐次分数差：", long_gap)
print(
    "湿日雨量分布使用全年回退的城市月份组：",
    mine_probability.fallback_count,
    "/",
    12 * len(PANEL)
)

2024：概率模型 + 历史湿日雨量分布
CRPS 6h     0.074936
CRPS 48h    0.080759
score       0.077847
相对云量模型：-0.0014723 ± 0.0000968
＋48 h 最大逐次分数差： 0.0
湿日雨量分布使用全年回退的城市月份组： 4 / 540


In [23]:
# 重新拟合：训练发布时刻及其标签都位于 2025 年以前
TRAIN_FULL = issue_times(0, SPLIT)
assert TRAIN_FULL.max() + 48 < SPLIT

feat_full = np.concatenate(
    [features_from_history(window(i), hours[i]) for i in TRAIN_FULL],
    axis=0
)
wet_full = (
    rain[TRAIN_FULL + 6].reshape(-1) >= 0.1
).astype(np.int8)

# 完全沿用 2024 验证时选定的参数，不根据 TEST 调参
wet_model_full = LGBMClassifier(**wet_model.get_params())
wet_model_full.fit(
    feat_full,
    wet_full,
    categorical_feature=[0]
)

# +48 h 的保留模型也重新拟合到 2024 年末
cloud_full = RainAndCloudClimatology(SPLIT, TRAIN_FULL)

agent_full = ProbabilityRainAgent(
    base_agent=cloud_full,
    wet_model=wet_model_full,
    train_end=SPLIT
)

# 在正式回放前，先核对一次输出格式
first_request = make_request(TEST[0])
check(agent_full.predict(first_request), first_request)

print("重新训练的发布时刻数：", len(TRAIN_FULL))
print("分类训练行数：", len(feat_full))
print("开始 2025—2026 回放……")

runs_full = evaluate(agent_full, TEST)
print("\n2025—2026 模型结果：")
print(runs_full.mean().round(6).to_string())

difference, se = compare(runs_full, runs_clim)
print(f"相对同场气候基线：{difference:+.7f} ± {se:.7f}")

print(
    "湿日雨量分布使用全年回退的城市月份组：",
    agent_full.fallback_count,
    "/",
    12 * len(PANEL)
)

重新训练的发布时刻数： 7293
分类训练行数： 328185
开始 2025—2026 回放……

2025—2026 模型结果：
CRPS 6h     0.069559
CRPS 48h    0.075750
score       0.072655
相对同场气候基线：-0.0032400 ± 0.0001898
湿日雨量分布使用全年回退的城市月份组： 3 / 540


In [24]:
import joblib

# 独立 agent 所需的 +48 h 气候分位数表
q48 = Climatology(SPLIT).table

# 独立 agent 所需的 +6 h 条件湿日雨量分位数表
q6_wet = np.stack([
    np.stack([
        agent_full.wet_quantiles[(m, city)]
        for city in range(len(PANEL))
    ])
    for m in range(1, 13)
])

assert q48.shape == (12, len(PANEL), 100)
assert q6_wet.shape == (12, len(PANEL), 100)

# 核对保存用的 +48 h 表确实复现当前模型
sample_request = make_request(TEST[0])
target48 = pd.Timestamp(sample_request["issue_time"]) + pd.Timedelta(hours=48)

np.testing.assert_allclose(
    q48[target48.month - 1],
    np.asarray(cloud_full.predict(sample_request)["rain"])[:, 1, :],
    rtol=0,
    atol=1e-10
)

MODEL_FILE = "rain_model.joblib"
joblib.dump(
    {
        "wet_model": wet_model_full,
        "q6_wet": q6_wet,
        "q48": q48,
        "levels": LEVELS,
        "cities": list(PANEL),
    },
    MODEL_FILE,
    compress=3
)

print("＋48 h 分位数表核对：通过")
print("已保存：", MODEL_FILE)
print("城市数：", len(PANEL), "；每个预测的成员数：", len(LEVELS))

＋48 h 分位数表核对：通过
已保存： rain_model.joblib
城市数： 45 ；每个预测的成员数： 100


In [26]:
%%writefile agent.py
"""European rain forecast: probability model and conditional rain quantiles."""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd


class Agent:
    def __init__(self):
        path = Path(__file__).resolve().parent / "rain_model.joblib"
        saved = joblib.load(path)

        self.wet_model = saved["wet_model"]
        self.q6_wet = np.asarray(saved["q6_wet"], dtype=float)
        self.q48 = np.asarray(saved["q48"], dtype=float)
        self.levels = np.asarray(saved["levels"], dtype=float)
        self.city_to_index = {
            (name, country): i
            for i, (name, country) in enumerate(saved["cities"])
        }

    def _features(self, request, issue, city_ids):
        history = np.nan_to_num(
            np.asarray(request["history"], dtype=float),
            nan=0.0, posinf=0.0, neginf=0.0
        )
        columns = {
            name: i
            for i, name in enumerate(request["feature_names"])
        }

        rain_history = history[:, :, columns["rain"]]
        cloud_history = history[:, :, columns["clouds"]]
        n_cities = len(city_ids)

        target = issue + pd.Timedelta(hours=6)
        season_angle = 2 * np.pi * (target.dayofyear - 1) / 365.25
        hour_angle = 2 * np.pi * target.hour / 24

        features = np.column_stack([
            city_ids,
            np.full(n_cities, np.sin(season_angle)),
            np.full(n_cities, np.cos(season_angle)),
            np.full(n_cities, np.sin(hour_angle)),
            np.full(n_cities, np.cos(hour_angle)),
            rain_history[:, -1],
            rain_history[:, -2],
            rain_history[:, -3],
            rain_history[:, -3:].sum(axis=1),
            rain_history[:, -6:].sum(axis=1),
            rain_history[:, -24:].sum(axis=1),
            rain_history.sum(axis=1),
            rain_history[:, -1] - rain_history[:, -7],
            cloud_history[:, -1],
            cloud_history[:, -1] - cloud_history[:, -7],
            history[:, -1, columns["humidity"]],
            history[:, -1, columns["temperature"]],
            history[:, -1, columns["wind_speed"]],
        ])

        return features.astype(np.float32)

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        city_ids = np.asarray([
            self.city_to_index[(city["name"], city["country"])]
            for city in request["cities"]
        ], dtype=int)

        features = self._features(request, issue, city_ids)
        named_features = pd.DataFrame(
            features,
            columns=self.wet_model.feature_name_
        )
        p_wet = self.wet_model.predict_proba(named_features)[:, 1]

        members = np.empty(
            (len(city_ids), len(request["horizons"]), len(self.levels)),
            dtype=float
        )

        for horizon_index, horizon in enumerate(request["horizons"]):
            target_month = (
                issue + pd.Timedelta(hours=int(horizon))
            ).month

            if horizon == 48:
                members[:, horizon_index, :] = self.q48[
                    target_month - 1, city_ids, :
                ]

            elif horizon == 6:
                for row, (city_id, probability) in enumerate(
                    zip(city_ids, p_wet)
                ):
                    probability = float(np.clip(probability, 0.0, 1.0))
                    values = np.zeros(len(self.levels), dtype=float)

                    if probability > 0:
                        wet_members = self.levels > (1.0 - probability)
                        conditional_levels = (
                            self.levels[wet_members] - (1.0 - probability)
                        ) / probability

                        values[wet_members] = np.interp(
                            conditional_levels,
                            self.levels,
                            self.q6_wet[target_month - 1, city_id]
                        )

                    members[row, horizon_index, :] = values

            else:
                raise ValueError(f"Unsupported horizon: {horizon}")

        return {"rain": members.tolist()}

Writing agent.py


In [27]:
import importlib
import agent

importlib.reload(agent)
standalone = agent.Agent()

for i in (TEST[0], TEST[len(TEST) // 2], TEST[-1]):
    request = make_request(i)

    expected = check(agent_full.predict(request), request)
    actual = check(standalone.predict(request), request)

    np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-10)

print("agent.py 与回放模型的三次预测一致：通过")
print("输出形状：", actual.shape)

# 再检查请求中的城市顺序改变时不会错位
reordered = dict(request)
reordered["cities"] = request["cities"][::-1]
reordered["history"] = request["history"][::-1]

reordered_actual = check(standalone.predict(reordered), reordered)
np.testing.assert_allclose(
    reordered_actual,
    actual[::-1],
    rtol=0,
    atol=1e-10
)
print("城市顺序变化检查：通过")

agent.py 与回放模型的三次预测一致：通过
输出形状： (45, 2, 100)
城市顺序变化检查：通过


In [28]:
import os

assert os.path.isfile("agent.py")
assert os.path.isfile("rain_model.joblib")

result = client.submit(
    CHALLENGE_ID,
    files=["agent.py", "rain_model.joblib"],
    submission_name="rain-hurdle-lightgbm-v1",
    runtime_id=182
)

submission_id = result["submission_id"]
print("新提交 ID：", submission_id)
print("提交结果：", result)

新提交 ID： 9068
提交结果： {'submission_id': 9068, 'deploy': {'deployment_id': 8780, 'is_deployable': False, 'is_settled': False, 'is_uploadable': False, 'last_status_message': 'Deployment queued successfully', 'message': 'Deployment started successfully', 'phase': 'deployment', 'status': 'deploy_queue', 'status_update_ts': '2026-09-28T15:19:29.347536Z'}}


In [33]:
status = client.status(submission_id, CHALLENGE_ID)

print("状态：", status.get("status"))
print("部署消息：", status.get("last_status_message"))
print("最近运行：")
for run in status.get("run_info", {}).get("results", [])[-5:]:
    print(
        run.get("created_at_ts"),
        run.get("job_status"),
        run.get("submission_reward")
    )

状态： active
部署消息： Deployment completed successfully
最近运行：
2026-09-28T15:19:29.266248Z completed None
2026-09-28T15:19:29.266248Z completed None


In [34]:
from pprint import pprint

submission_id = 9068
status = client.status(submission_id, CHALLENGE_ID)
pprint(status)

{'challenge_id': 177,
 'created_at_ts': '2026-09-28T15:19:24.953909Z',
 'id': 9068,
 'is_deployable': False,
 'is_owner': True,
 'is_public': False,
 'is_settled': True,
 'is_uploadable': False,
 'last_status_message': 'Deployment completed successfully',
 'latest_deploy': {'created_at_ts': '2026-09-28T15:19:29.266248Z',
                   'failure_message': None,
                   'finished_at_ts': '2026-09-28T15:21:27.861743Z',
                   'id': 8780,
                   'status': 'succeeded'},
 'phase': 'active',
 'queue_info': {'created_at_ts': None,
                'in_queue_for_second': None,
                'queue_position': None,
                'queue_total': None},
 'run_info': {'frontend_precision': 4,
              'has_gpu': False,
              'is_elo_score': False,
              'metric': 'Mean CRPS (mm)',
              'number_of_agents': 1,
              'results': [{'created_at_ts': '2026-09-28T15:19:29.266248Z',
                           'env_error_message':